# AGO1 (AGO1 (A)) / FBW2 (B)) / miR165a (C)) Domain Contact Analysis

**Kernel:** `abcfold-ago1-mirs-notebook` (`envs/notebook.yaml`) — install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-ago1-mirs-notebook
python -m ipykernel install --user --name abcfold-ago1-mirs-notebook
```

PLIP contacts from `results/ago1_fbw2_mir165a/all_selected_summary.csv`, produced by
`worflows/postprocessing/Snakefile` (stage 3f `run_plip` + 3h `aggregate`).
Same structure as the sibling DRB2 pipeline's `notebooks/rna_ds_drb2_drb4_domain_analysis.ipynb`:
pooled / per-pose-cluster / per-backend domain-contact heatmaps + a
residue-level interface map.

This complex has a miR RNA chain. The **main** PLIP pass (`plip.chains = [['A'], ['B', 'C']]`, `dnareceptor: true`) folds the miR into the *receptor* group alongside AGO1, so it only surfaces AGO1-vs-protein-partner contacts. The per-nucleotide miR view comes from the separate `plip_mir_ligands` pass — load `all_selected_summary_mir_ligands.csv` (section at the end) for that.

**Receptor** is always the pose-cluster anchor, AGO1 (chain A). Every
heatmap is one *couple*: AGO1 vs. one partner chain.

---

**Domain boundaries below are PROSITE / InterPro-verified** (ScanProsite +
InterProScan 5, scanned against the exact folded sequences on 2026-08-28,
cross-checked against UniProt — see the provenance block in
`notebooks/generate_domain_notebooks.py`). PROSITE profile spans are quoted
verbatim where one exists (AGO1 PAZ/PIWI, CUL1 cullin homology). Set
`USE_WINDOWS = True` in the domain-definitions cell to fall back to plain
fixed-width residue windows instead (every heatmap still renders, axis
labels just become `1-60`, `61-120`, …).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "ago1_fbw2_mir165a"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "AGO1"
LIGAND_CHAIN_NAMES = {"B": "FBW2", "C": "miR165a"}
PROTEIN_PARTNERS = {"B": "FBW2"}
RNA_PARTNERS = {"C": "miR165a"}
CHAIN_LENGTHS = {"A": 1050, "B": 317, "C": 21}
RECEPTOR_LEN = 1050
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
BACKEND_PALETTE = px.colors.qualitative.Set2
ITYPE_PALETTE = px.colors.qualitative.Set1

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()

## Load data

In [2]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)
df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv entry")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{csv_path}: {len(df)} contact rows, {df['cluster'].nunique()} pose cluster(s), "
      f"{n_models} model(s), before the energy filter below")
print("\ncontact rows per backend:")
print(df.groupby("backend").size().rename("rows").to_frame())

../results/ago1_fbw2_mir165a/all_selected_summary.csv: 42816 contact rows, 6 pose cluster(s), 500 model(s), before the energy filter below

contact rows per backend:
               rows
backend            
alphafold3     5893
chai1         10408
openfold3      8414
protenix       7241
rosettafold3  10860


## Filter out numerically-unconverged structures

Robust (median / MAD) modified z-score on each minimised structure's final
ChimeraX energy — same approach as the DRB2 pipeline's notebooks.

1. **Whole-backend cut:** if a backend has >50 % of its energy-assessed models
   flagged as outliers, the *entire* backend is dropped.
2. **Missing `_energy.csv` ⇒ keep:** minimisations that produced a `*_fixed.pdb`
   but no parseable energy trajectory are kept ("not assessed", not "bad") for
   a non-dropped backend.


In [3]:
def read_final_energy(energy_csv_path):
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)
        for line in fh:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue
            try:
                last_energy = float(parts[1])
            except ValueError:
                pass
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber", "_nonprot")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                        "final_energy": read_final_energy(energy_csv)})

energy_all = pd.DataFrame(energy_rows).merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left")
energy_df = energy_all.dropna(subset=["final_energy"]).copy()

MOD_Z_THRESHOLD = 3.5
pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(energy_df.groupby("backend")["final_energy"].agg(["count", "min", "median", "max"]).round(1).to_string())

flag_rate = (1 - energy_df.groupby("backend")["energy_ok"].mean()).rename("flagged_frac")
DROP_BACKENDS = sorted(flag_rate[flag_rate > 0.5].index)
print(f"\nflagged fraction by backend (|mod-z| > {MOD_Z_THRESHOLD}):")
print(flag_rate.round(3).to_string())
print(f"\n=> whole-backend drop (>50% flagged): {DROP_BACKENDS or 'none'}")

Pooled final-energy median=-190,133 kJ/mol, MAD=6,227
              count       min    median           max
backend                                              
alphafold3       95 -184441.2 -179023.6 -1.703798e+05
chai1            90 -199507.0 -196354.5 -1.856897e+05
openfold3        94 -198179.8 -192259.1 -1.802554e+05
protenix         89 -199050.8 -192582.1 -1.758267e+05
rosettafold3     86 -195273.7 -182935.3  1.220518e+21

flagged fraction by backend (|mod-z| > 3.5):
backend
alphafold3      0.000
chai1           0.000
openfold3       0.000
protenix        0.000
rosettafold3    0.291

=> whole-backend drop (>50% flagged): none


In [4]:
assessed_ok = set(zip(energy_df.loc[energy_df["energy_ok"], "cluster"],
                      energy_df.loc[energy_df["energy_ok"], "fname"]))
unassessed = set(zip(energy_all.loc[energy_all["final_energy"].isna(), "cluster"],
                     energy_all.loc[energy_all["final_energy"].isna(), "fname"]))
keep_pairs = assessed_ok | unassessed

drop_fnames = set(sel.loc[sel["backend"].isin(DROP_BACKENDS), "fname"])
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups

df = df[df_keys.isin(keep_pairs) & ~df["fname"].isin(drop_fnames)].copy()

n_models_after = df.groupby(["cluster", "fname"]).ngroups
print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
print(df.groupby("backend")["fname"].nunique().rename("n_models").to_frame())

475 / 500 model(s) kept after the energy filter
              n_models
backend               
alphafold3         100
chai1              100
openfold3           99
protenix           100
rosettafold3        75


## Domain definitions

`CURATED_DOMAINS` below is the PROSITE / InterPro-verified table (see the
header note above). `USE_WINDOWS = True` ignores it and bins every chain
into fixed-width residue windows instead.

In [5]:
USE_WINDOWS = False        # True -> fixed-width residue windows instead of CURATED_DOMAINS
WINDOW = 60               # residues per window when USE_WINDOWS

# PROSITE / InterPro-verified boundaries (1-based inclusive), scanned
# against the exact folded sequences 2026-08-28. See generate_domain_notebooks.py.
CURATED_DOMAINS = {
    "FBW2": [
        ("F_box", 1, 54),
        ("LRR_solenoid", 55, 244),
        ("C_tail", 245, 317),
    ],
    "AGO1": [
        ("N_ext_Grich", 1, 189),
        ("ArgoN", 190, 335),
        ("ArgoL1", 336, 389),
        ("PAZ", 390, 503),
        ("ArgoL2", 504, 585),
        ("MID", 586, 677),
        ("PIWI", 678, 1050),
    ],
}

def windows_for(chain_len, w=None):
    w = w or WINDOW
    return [(f"{s}-{min(s+w-1, chain_len)}", s, min(s + w - 1, chain_len))
            for s in range(1, chain_len + 1, w)]

def domains_for(name, chain_len):
    if USE_WINDOWS or name not in CURATED_DOMAINS:
        return windows_for(chain_len)
    return CURATED_DOMAINS[name]

def make_domain_mapper(domain_ranges):
    intervals = pd.IntervalIndex.from_tuples(
        [(s, e) for _, s, e in domain_ranges], closed="both")
    labels = [l for l, _, _ in domain_ranges]
    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series([labels[i] if i != -1 else pd.NA for i in idx],
                         index=resnr_series.index, dtype="object")
    return mapper

RECEPTOR_DOMAINS = domains_for(RECEPTOR_NAME, RECEPTOR_LEN)
RECEPTOR_LABELS = [l for l, _, _ in RECEPTOR_DOMAINS]
receptor_mapper = make_domain_mapper(RECEPTOR_DOMAINS)

LIGAND_DOMAINS = {c: domains_for(nm, CHAIN_LENGTHS[c]) for c, nm in PROTEIN_PARTNERS.items()}
LIGAND_LABELS = {c: [l for l, _, _ in d] for c, d in LIGAND_DOMAINS.items()}
LIGAND_MAPPERS = {c: make_domain_mapper(d) for c, d in LIGAND_DOMAINS.items()}

## Chain-residue offset detection

After `fix_pdb` (pdb4amber + PDBFixer) the whole complex is usually renumbered
continuously across chains rather than restarting each chain at 1. This cell
auto-detects the per-chain offset: for each ligand chain it finds the offset
that lands every `resnr_lig` inside `[1, chain_length]`, so domain mapping is
on that chain's own numbering. (If the receptor `resnr` isn't already 1-based
it's offset too.)

In [6]:
def best_offset(values, chain_len):
    v = values.dropna().astype(int)
    if v.empty:
        return 0
    cand = int(v.min()) - 1
    for off in (0, cand):
        if v.sub(off).between(1, chain_len).all():
            return off
    return cand

rec_off = best_offset(df["resnr"], RECEPTOR_LEN)
df["resnr_raw"] = df["resnr"]
df["resnr"] = df["resnr"] - rec_off
df["receptor_domain"] = receptor_mapper(df["resnr"])

df["resnr_lig_raw"] = df["resnr_lig"]
CHAIN_OFFSET = {}
for chain, clen in CHAIN_LENGTHS.items():
    if chain == RECEPTOR_CHAIN:
        continue
    mask = df["reschain_lig"] == chain
    if not mask.any():
        continue
    off = best_offset(df.loc[mask, "resnr_lig"], clen)
    CHAIN_OFFSET[chain] = off
    df.loc[mask, "resnr_lig"] = df.loc[mask, "resnr_lig"] - off
    bad = df.loc[mask & ~df["resnr_lig"].between(1, clen)]
    tag = "OK" if bad.empty else f"WARNING {len(bad)} rows outside [1,{clen}]"
    print(f"chain {chain} ({LIGAND_CHAIN_NAMES.get(chain, '?')}): offset {off} -> {tag}")

df["ligand_domain"] = pd.NA
for chain, mapper in LIGAND_MAPPERS.items():
    m = df["reschain_lig"] == chain
    df.loc[m, "ligand_domain"] = mapper(df.loc[m, "resnr_lig"])

n_unmapped = df["receptor_domain"].isna().sum()
print(f"\nUnmapped {RECEPTOR_NAME} residues: {n_unmapped} ({100*n_unmapped/max(len(df),1):.1f}%)")

chain B (FBW2): offset 1050 -> OK



Unmapped AGO1 residues: 0 (0.0%)


## Interactor couples in this dataset

One heatmap per couple: AGO1 (fixed receptor) vs. each partner. Couples with
zero rows are expected for chains folded into the PLIP receptor group — the
check below makes that explicit.

In [7]:
COUPLES = ["B", "C"]

print("Contact rows per couple (AGO1 vs. partner):")
for c in COUPLES:
    n = (df["reschain_lig"] == c).sum()
    n_models_c = df.loc[df["reschain_lig"] == c, ["cluster", "fname"]].drop_duplicates().shape[0]
    status = "POPULATED" if n else "EMPTY -- folded into the PLIP receptor group or no contact"
    print(f"  {RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]:8s}: {n:6d} rows / {n_models_c:3d} model(s) -- {status}")

Contact rows per couple (AGO1 vs. partner):
  AGO1 x FBW2    :  39468 rows / 475 model(s) -- POPULATED
  AGO1 x miR165a :      0 rows /   0 model(s) -- EMPTY -- folded into the PLIP receptor group or no contact


In [8]:
def domain_pair_matrix(data, ligand_chain, n_models_norm):
    """(x_labels, y_labels, rate matrix) for a protein couple; None if empty."""
    labels = LIGAND_LABELS[ligand_chain]
    sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["receptor_domain", "ligand_domain"])
    if sub.empty:
        return None
    ct = (sub.groupby(["receptor_domain", "ligand_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=RECEPTOR_LABELS, columns=labels, fill_value=0))
    return labels, RECEPTOR_LABELS, (ct / n_models_norm if n_models_norm else ct)

def rna_pos_matrix(data, ligand_chain, n_models_norm):
    sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["receptor_domain"])
    if sub.empty:
        return None
    nt = sorted(sub["resnr_lig"].dropna().unique())
    ct = (sub.groupby(["resnr_lig", "receptor_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=nt, columns=RECEPTOR_LABELS, fill_value=0))
    return RECEPTOR_LABELS, [str(int(p)) for p in nt], (ct / n_models_norm if n_models_norm else ct)

def couple_matrix(data, ligand_chain, n_models_norm):
    if ligand_chain in RNA_PARTNERS:
        return rna_pos_matrix(data, ligand_chain, n_models_norm)
    return domain_pair_matrix(data, ligand_chain, n_models_norm)

def heatmap_from_matrix(res, ligand_chain, title, filename, subdir="domain_contacts"):
    if res is None:
        print(f"No contacts -- skipping '{title}'.")
        return
    x_labels, y_labels, rate = res
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    is_rna = ligand_chain in RNA_PARTNERS
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=x_labels, y=y_labels, colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=8),
        hovertemplate=(f"{RECEPTOR_NAME}: %{{{'x' if is_rna else 'y'}}}<br>{ligand_name}: "
                       f"%{{{'y' if is_rna else 'x'}}}<br>%{{z:.3f}} contacts/model<extra></extra>"),
        colorbar=dict(title="Mean<br>contacts/<br>model")))
    fig.update_layout(
        title=title,
        xaxis_title=f"{RECEPTOR_NAME} domain" if is_rna else f"{ligand_name} domain",
        yaxis_title=f"{ligand_name} nt position" if is_rna else f"{RECEPTOR_NAME} domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(560, len(x_labels) * 90), height=max(420, len(y_labels) * (16 if is_rna else 64)))
    save_fig(fig, filename, subdir)

## One heatmap per couple (all backends, all clusters pooled)

In [9]:
n_models_total = df.groupby(["cluster", "fname"]).ngroups
for c in COUPLES:
    heatmap_from_matrix(
        couple_matrix(df, c, n_models_total), c,
        f"{RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]} -- pooled (n={n_models_total} models)",
        f"{RECEPTOR_NAME.lower()}_{LIGAND_CHAIN_NAMES[c].lower()}_heatmap_pooled.html")

Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/domain_contacts/ago1_fbw2_heatmap_pooled.html


No contacts -- skipping 'AGO1 x miR165a -- pooled (n=475 models)'.


## Per-pose-cluster breakdown

One small-multiple panel per couple: one heatmap per pose cluster (`scripts/pose_cluster_anchor.py`'s `cluster` column), shared colour scale
within the panel. If there's only one cluster this collapses to a single tile.

In [10]:
clusters = sorted(df["cluster"].unique())
cluster_n_models = df.groupby("cluster")["fname"].nunique()
print(f"{len(clusters)} pose cluster(s): " + ", ".join(f'{k} (n={cluster_n_models[k]})' for k in clusters))

def panel_by(group_col, group_values, group_n_models, tag, subdir):
    for c in COUPLES:
        ligand_name = LIGAND_CHAIN_NAMES[c]
        mats = {}
        for g in group_values:
            res = couple_matrix(df[df[group_col] == g], c, group_n_models[g])
            if res is not None:
                mats[g] = res
        if not mats:
            print(f"No {RECEPTOR_NAME}-{ligand_name} contacts for any {tag} -- skipping.")
            continue
        present = list(mats)
        zmax = max(r[2].values.max() for r in mats.values()) or 1
        ncols = min(3, len(present)); nrows = -(-len(present) // ncols)
        fig = make_subplots(rows=nrows, cols=ncols,
                            subplot_titles=[f"{tag} {g} (n={group_n_models[g]})" for g in present])
        for i, g in enumerate(present):
            x_labels, y_labels, rate = mats[g]
            r, cc = i // ncols + 1, i % ncols + 1
            fig.add_trace(go.Heatmap(z=rate.values, x=x_labels, y=y_labels, colorscale="YlOrRd",
                zmin=0, zmax=float(zmax), showscale=bool(g == present[-1]),
                text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
                texttemplate="%{text}", textfont=dict(size=7),
                hovertemplate=f"{tag} {g}<br>%{{y}} / %{{x}}<br>%{{z:.3f}}/model<extra></extra>"),
                row=r, col=cc)
        fig.update_yaxes(autorange="reversed")
        fig.update_layout(title=f"{RECEPTOR_NAME} x {ligand_name} by {tag}", template=TEMPLATE,
                          width=max(760, 360 * ncols), height=max(430, 150 * nrows))
        save_fig(fig, f"{RECEPTOR_NAME.lower()}_{ligand_name.lower()}_heatmap_by_{tag}.html", subdir)

panel_by("cluster", clusters, cluster_n_models, "cluster", "per_cluster")

6 pose cluster(s): 1 (n=124), 2 (n=232), 3 (n=51), 4 (n=15), 5 (n=52), 6 (n=1)
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/per_cluster/ago1_fbw2_heatmap_by_cluster.html


No AGO1-miR165a contacts for any cluster -- skipping.


## Per-backend breakdown

Cross-architecture agreement check: do the six ABCfold backends place the
same domain–domain contacts? One panel per couple, one heatmap per surviving
backend, shared colour scale.

In [11]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend")["fname"].nunique()
print(f"{len(backends)} backend(s): " + ", ".join(f'{b} (n={backend_n_models[b]})' for b in backends))
panel_by("backend", backends, backend_n_models, "backend", "per_backend")

5 backend(s): alphafold3 (n=100), chai1 (n=100), openfold3 (n=99), protenix (n=100), rosettafold3 (n=75)
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/per_backend/ago1_fbw2_heatmap_by_backend.html


No AGO1-miR165a contacts for any backend -- skipping.


## Interaction-type breakdown

In [12]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

print("\nInteraction type by receptor-ligand chain pair:")
display(df.groupby(["reschain", "reschain_lig", "interaction_type"], observed=True)
        .size().rename("count").reset_index().sort_values("count", ascending=False).head(20))

per_backend_itype = (df.groupby(["backend", "interaction_type"], observed=True).size()
    .div(df.groupby("backend")["fname"].nunique(), level="backend")
    .rename("contacts_per_model").reset_index())
fig = px.bar(per_backend_itype, x="backend", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=ITYPE_PALETTE, template=TEMPLATE,
             title=f"{RECEPTOR_NAME} interface contacts per model, by interaction type and backend")
fig.update_layout(width=780, height=460, xaxis_title="", yaxis_title="contacts / model")
save_fig(fig, "interaction_types_by_backend.html")

Interaction type counts:


,count
interaction_type,
hydrogen_bonds,18976
hydrophobic_interactions,11977
salt_bridges,7336
pi-cation_interactions,951
pi-stacking,228



Interaction type by receptor-ligand chain pair:


,reschain,reschain_lig,interaction_type,count
0,A,B,hydrogen_bonds,18976
1,A,B,hydrophobic_interactions,11977
4,A,B,salt_bridges,7336
2,A,B,pi-cation_interactions,951
3,A,B,pi-stacking,228


Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/interaction_types_by_backend.html


## Residue-level interface map

Single-residue resolution for the busiest AGO1 and partner residues, split by
interaction type, plus the 2-D AGO1-residue × partner-residue contact map —
the predicted interface footprint, pooled across surviving backends. Runs for
the first populated protein couple; edit `COUPLE_FOR_RESIDUE_MAP` to switch.

In [13]:
COUPLE_FOR_RESIDUE_MAP = 'B'   # a protein partner chain letter
TOP_N = 25
bc = df[df["reschain_lig"] == COUPLE_FOR_RESIDUE_MAP].copy()
n_norm = df.groupby(["cluster", "fname"]).ngroups
partner_name = LIGAND_CHAIN_NAMES.get(COUPLE_FOR_RESIDUE_MAP, "partner")

def residue_rate_bar(data, resnr_col, restype_col, chain_label, filename):
    if data.empty:
        print(f"{chain_label}: no rows -- skipping."); return pd.DataFrame({resnr_col: [], restype_col: []})
    per_res_total = (data.groupby([resnr_col, restype_col], observed=True).size()
                     .div(n_norm).rename("rate").reset_index()
                     .sort_values("rate", ascending=False).head(TOP_N))
    order = per_res_total[resnr_col].tolist()
    per_res_itype = (data.groupby([resnr_col, restype_col, "interaction_type"], observed=True).size()
                     .div(n_norm).rename("rate").reset_index())
    per_res_itype = per_res_itype[per_res_itype[resnr_col].isin(order)].copy()
    per_res_itype["label"] = (per_res_itype[restype_col].astype(str)
                              + per_res_itype[resnr_col].astype(int).astype(str))
    label_order = [f"{per_res_total.loc[per_res_total[resnr_col] == r, restype_col].iloc[0]}{int(r)}"
                   for r in order]
    fig = px.bar(per_res_itype, x="label", y="rate", color="interaction_type",
                 category_orders={"label": label_order}, color_discrete_sequence=ITYPE_PALETTE,
                 template=TEMPLATE, title=f"{chain_label} interface residues -- top {TOP_N} by contact rate (n={n_norm})")
    fig.update_layout(width=950, height=460, xaxis_title=f"{chain_label} residue",
                      yaxis_title="contacts / model", xaxis_tickangle=-45)
    save_fig(fig, filename, "residue_interface")
    return per_res_total

top_rec = residue_rate_bar(bc, "resnr", "restype", RECEPTOR_NAME, f"{RECEPTOR_NAME.lower()}_top_residues.html")
top_lig = residue_rate_bar(bc, "resnr_lig", "restype_lig", partner_name, f"{partner_name.lower()}_top_residues.html")
display(top_rec.reset_index(drop=True)); display(top_lig.reset_index(drop=True))

Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/residue_interface/ago1_top_residues.html


Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/residue_interface/fbw2_top_residues.html


,resnr,restype,rate
0,474,ARG,1.555789
1,907,ARG,1.275789
2,716,LYS,1.037895
3,418,LYS,0.920000
4,421,ARG,0.848421
5,720,GLN,0.842105
6,3,ARG,0.797895
7,473,PHE,0.791579
8,469,GLU,0.772632
9,712,LYS,0.760000


,resnr_lig,restype_lig,rate
0,295,TRP,2.018947
1,252,TYR,1.997895
2,313,TRP,1.850526
3,292,GLU,1.549474
4,288,GLU,1.515789
5,300,TYR,1.475789
6,265,TYR,1.448421
7,275,TRP,1.444211
8,297,ASP,1.433684
9,140,TYR,1.364211


In [14]:
if not bc.empty and len(top_rec) and len(top_lig):
    tr = top_rec["resnr"].tolist(); tl = top_lig["resnr_lig"].tolist()
    pair = bc[bc["resnr"].isin(tr) & bc["resnr_lig"].isin(tl)]
    mat = (pair.groupby(["resnr", "resnr_lig"], observed=True).size()
           .div(n_norm).unstack(fill_value=0)
           .reindex(index=sorted(tr), columns=sorted(tl), fill_value=0))
    rtick = {int(r): f"{bc.loc[bc['resnr']==r,'restype'].iloc[0]}{int(r)}" for r in mat.index}
    ltick = {int(c): f"{bc.loc[bc['resnr_lig']==c,'restype_lig'].iloc[0]}{int(c)}" for c in mat.columns}
    fig = go.Figure(go.Heatmap(z=mat.values, x=[ltick[c] for c in mat.columns], y=[rtick[r] for r in mat.index],
        colorscale="YlOrRd", hovertemplate=f"{RECEPTOR_NAME} %{{y}}<br>{partner_name} %{{x}}<br>%{{z:.3f}}/model<extra></extra>",
        colorbar=dict(title="contacts<br>/ model")))
    fig.update_layout(title=f"{RECEPTOR_NAME} x {partner_name} residue-residue contact map -- top {TOP_N} each (n={n_norm})",
        xaxis_title=f"{partner_name} residue", yaxis_title=f"{RECEPTOR_NAME} residue",
        yaxis=dict(autorange="reversed"), template=TEMPLATE, width=900, height=760)
    save_fig(fig, f"{RECEPTOR_NAME.lower()}_{partner_name.lower()}_residue_contact_map.html", "residue_interface")

Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/residue_interface/ago1_fbw2_residue_contact_map.html


## miR-ligand PLIP pass (per-nucleotide)

`results/ago1_fbw2_mir165a/all_selected_summary_mir_ligands.csv` — the second
PLIP pass with `--chains` omitted and no `--dnareceptor`, so every miR
nucleotide is reported as its own ligand. Here we filter to AGO1 (chain A)
contacts and map each nucleotide position × AGO1 domain.

In [15]:
mir_csv = RESULTS_DIR / "all_selected_summary_mir_ligands.csv"
if mir_csv.exists():
    mdf = pd.read_csv(mir_csv).rename(columns={"replica": "cluster", "model": "fname"})
    mdf = mdf.merge(sel, on=["fname"], how="left", suffixes=("", "_sel"))
    # keep rows where AGO1 (chain A) is on either side of the contact
    a_side = mdf[mdf["reschain"] == "A"].copy()
    a_side["ago1_domain"] = receptor_mapper(a_side["resnr"] - rec_off)
    nt = sorted(a_side["resnr_lig"].dropna().unique())
    n_norm_m = a_side.groupby(["cluster", "fname"]).ngroups or 1
    ct = (a_side.dropna(subset=["ago1_domain"]).groupby(["resnr_lig", "ago1_domain"], observed=True)
          .size().div(n_norm_m).unstack(fill_value=0)
          .reindex(index=nt, columns=RECEPTOR_LABELS, fill_value=0))
    fig = go.Figure(go.Heatmap(z=ct.values, x=RECEPTOR_LABELS, y=[str(int(p)) for p in nt],
        colorscale="YlOrRd", colorbar=dict(title="contacts<br>/ model"),
        hovertemplate="AGO1 domain %{x}<br>miR nt %{y}<br>%{z:.3f}/model<extra></extra>"))
    fig.update_layout(title=f"miR nucleotide x AGO1 domain contact rate (n={n_norm_m} models)",
        xaxis_title="AGO1 domain", yaxis_title="miR nt position",
        yaxis=dict(autorange="reversed"), template=TEMPLATE, width=760, height=max(400, len(nt) * 18))
    save_fig(fig, "mir_nt_x_ago1_domain_heatmap.html", "mir_ligands")
else:
    print(f"{mir_csv} not found -- run the plip_mir_ligands pass first.")

Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/mir_ligands/mir_nt_x_ago1_domain_heatmap.html


## Export contact tables

In [16]:
export_dir = out_path("tables", "")
for c in COUPLES:
    for tag, data in [("pooled", df)] + [(b, df[df["backend"] == b]) for b in backends]:
        res = couple_matrix(data, c, data.groupby(["cluster", "fname"]).ngroups)
        if res is None:
            continue
        _, _, rate = res
        p = export_dir / f"{RECEPTOR_NAME.lower()}_{LIGAND_CHAIN_NAMES[c].lower()}_rate_{tag}.csv"
        rate.to_csv(p)
        print(f"Saved: {p}")

Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_pooled.csv
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_alphafold3.csv
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_chai1.csv
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_openfold3.csv
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_protenix.csv
Saved: ../results/ago1_fbw2_mir165a/figures/domain_analysis/tables/ago1_fbw2_rate_rosettafold3.csv
